# The Essence of Attention and Transformers
### Step-by-Step Demo: Following the Transformer Data Flow
Ref: Attention Is All You Need https://arxiv.org/abs/1706.03762

```
Text --> [1. Tokenize] --> [2. Embed] --> [3. + Position] --> [4. Attention] --> [5. Multi-Head] --> [6. Encoder] --> [7. Decoder]
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

---
## Step 1: Sequence and Tokenization

A tokenizer splits text into tokens (words or subwords) and maps each one
to a unique integer ID using a vocabulary lookup table.

In [ ]:
# 1a. Build a simple vocabulary

# A vocabulary is a mapping from words to integer IDs.
# Real tokenizers (like BPE or WordPiece) handle subwords

vocab = {
    '<PAD>': 0,    # padding token (used to fill short sequences)
    '<UNK>': 1,    # unknown token (for words not in vocab)
    'i'    : 2,
    'love' : 3,
    'ai'   : 4,
    'the'  : 5,
    'cat'  : 6,
    'sat'  : 7,
    'on'   : 8,
    'mat'  : 9,
}

vocab_size = len(vocab)
print(f'Vocabulary size: {vocab_size} words')
print(f'Vocabulary: {vocab}')

In [ ]:
# 1b. Tokenize a sentence

def tokenize(sentence, vocab):
    words = sentence.lower().split()
    return [vocab.get(w, vocab['<UNK>']) for w in words]

sentence = 'the cat sat on the mat'
token_ids = tokenize(sentence, vocab)

print(f'Input sentence: "{sentence}"')
print(f'Token IDs:      {token_ids}')
print()

words = sentence.lower().split()
for word, tid in zip(words, token_ids):
    print(f'  "{word}" --> {tid}')

print()
print(f'Sequence length: {len(token_ids)}')

---
## Step 2: Word Embedding


```
token ID 6 ('cat') --> [0.12, -0.45, 0.78, ...]   (d_model numbers)
token ID 7 ('sat') --> [0.33,  0.21, -0.55, ...]   (d_model numbers)
```

The embedding is simply a lookup table (a matrix of shape vocab_size x d_model).
Row i of the table is the vector for token ID i.
These vectors are learned during training.

In [ ]:
# 2a. Create an embedding table

d_model = 8   # embedding dimension (small for readability; paper uses 512)

# The embedding table: one row per word in the vocabulary.
# In a real model these values are learned. Here we use random initialization.
embedding_table = np.random.randn(vocab_size, d_model)

print(f'Embedding table shape: {embedding_table.shape}  (vocab_size x d_model)')
print(f'  {vocab_size} words, each represented by a {d_model}-dimensional vector.')
print()
print('Embedding table (each row is one word vector):')
print(np.round(embedding_table, 3))

In [ ]:
# 2b. Look up embeddings for our token sequence

# Embedding is just indexing into the table: take row [token_id]
token_ids_array = np.array(token_ids)
embedded_sequence = embedding_table[token_ids_array]   # shape: (seq_len, d_model)

print(f'Token IDs:  {token_ids}')
print(f'Words:      {words}')
print(f'Embedded sequence shape: {embedded_sequence.shape}  (seq_len x d_model)')
print()

for i, (word, tid) in enumerate(zip(words, token_ids)):
    vec = np.round(embedded_sequence[i], 3)
    print(f'  "{word}" (id={tid}) --> {vec}')

In [ ]:
# 2c. Visualize: same word = same vector

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(embedded_sequence, aspect='auto', cmap='RdYlBu')

ax.set_yticks(range(len(words)))
ax.set_yticklabels([f'{w} (id={t})' for w, t in zip(words, token_ids)])
ax.set_xlabel('Embedding Dimension')
ax.set_ylabel('Token in Sequence')
ax.set_title('Embedded Sequence\n'
             'Same word ("the") has identical rows')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## Step 3: Positional Encoding

Formula:
```
PE[pos, 2i]   = sin( pos / 10000^(2i / d_model) )
PE[pos, 2i+1] = cos( pos / 10000^(2i / d_model) )
```

In [ ]:
# 3a. Build the Positional Encoding matrix

def positional_encoding(max_len, d_model):
    PE  = np.zeros((max_len, d_model))
    for k in range(max_len):
        # Used for mapping to column indices 0 <= i < d_model/2, with a single value of i maps to both sine and cosine functions
        for i in np.arange(int(d_model/2)):
            denominator = np.power(10000, 2*i/d_model)
            PE[k, 2*i] = np.sin(k/denominator)
            PE[k, 2*i+1] = np.cos(k/denominator)
    return PE

# 50 and 128 servs for the demo to be easy for visual
MAX_LEN = 50
D_MODEL = 128

PE = positional_encoding(MAX_LEN, D_MODEL)

print('Positional Encoding shape:', PE.shape)
print(f'  {MAX_LEN} positions, each with a unique {D_MODEL}-dimensional fingerprint.')
print()
print('First 3 positions, first 8 dimensions:')
print(np.round(PE[:3, :8], 4))

In [ ]:
# 3b. Heatmap of the full PE matrix

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(PE, aspect='auto', cmap='RdYlBu', origin='upper')

ax.xaxis.set_label_position('top')
ax.xaxis.tick_top()
ax.set_xlabel('Embedding Dimension')

ax.set_ylabel('Position in Sequence')
ax.set_title('Positional Encoding Heatmap\nEach row is one position fingerprint', pad=20)

cbar = plt.colorbar(im, ax=ax)
cbar.ax.set_xlabel('value', labelpad=8)
cbar.ax.xaxis.set_label_position('bottom')

plt.tight_layout()
plt.show()

print('Left side (low dims): high frequency, captures fine position differences.')
print('Right side (high dims): low frequency, captures coarse position differences.')

In [ ]:
# 3d. Show how PE is added to embeddings

# Reuse the embedding from Step 2 (small example for readability)
sample_words = ['the', 'cat', 'sat', 'on', 'the', 'mat']
sample_seq_len = len(sample_words)

sample_embeddings = embedded_sequence
sample_pe = positional_encoding(sample_seq_len, d_model)

combined = sample_embeddings + sample_pe

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

titles = ['Token Embeddings\n(WHAT the word is)',
          'Positional Encoding\n(WHERE the word is)',
          'Combined = Embedding + PE\n(WHAT + WHERE)']
data   = [sample_embeddings, sample_pe, combined]

for ax, title, matrix in zip(axes, titles, data):
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlBu')
    ax.set_yticks(range(sample_seq_len))
    ax.set_yticklabels(sample_words)
    ax.set_xlabel('Dimension')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()

print('Left: same word "the" has identical rows (positions 0 and 4).')
print('Middle: positions 0 and 4 have different PE vectors.')
print('Right: after adding PE, the two "the" tokens are now distinguishable.')